In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Call external APIs from an agent

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

### Agents and existing systems

Store data lives in many systems: order management, workforce, merchandising, weather and more. An agent reaches them through tools. ADK gives you two common ways to do that.

### OpenAPI tools

If a system publishes an [OpenAPI](https://www.openapis.org/) description, ADK's [`OpenAPIToolset`](https://google.github.io/adk-docs/tools-custom/openapi-tools/) reads it and creates one tool per operation, with the parameters and descriptions from the document. You attach the credentials once, and you do not write a wrapper for each endpoint.

### Function tools over a public API

For a small public API, a function tool with an HTTP call is often simpler. This quickstart uses the keyless [Open-Meteo](https://open-meteo.com/) weather API.

### The order management API

This quickstart runs a small mock of an order management API for buy-online-pick-up-in-store (BOPIS) orders at the Naperville store. It needs an API key in the `X-API-Key` header. Ready orders carry a `hold_until` time, after which the order is cancelled and its items go back to stock.

<img width="60%" src="../../docs/diagrams/q04.png" alt="An agent that calls an order management API through OpenAPI tools and a weather API through a function tool" />

### Objectives

In this tutorial, you will learn how to connect an agent to external APIs.

You will complete the following tasks:

- Start the mock order management API and call it directly
- Generate tools from its OpenAPI description, with an API key
- Write a function tool over a public weather API
- Run an agent that uses both, with a plugin that lets the model retry a failed call

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI

Learn about [Gemini on Vertex AI pricing](https://cloud.google.com/vertex-ai/generative-ai/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This quickstart reads the store data you loaded during setup, in your own namespace. Set your project ID and the namespace you chose.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# This notebook sits two folders below the repository root, where the shared store tools live
REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the notebook output to the agent's own events: ADK marks experimental features with a
# UserWarning, SDKs announce renamed classes with a FutureWarning, and the Gen AI SDK logs a note
# whenever a response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("google_genai").setLevel(logging.ERROR)

import subprocess
import time

import httpx
from google.adk.agents import LlmAgent
from google.adk.apps import App
from google.adk.models import Gemini
from google.adk.plugins import ReflectAndRetryToolPlugin
from google.adk.runners import InMemoryRunner
from google.adk.tools.openapi_tool import OpenAPIToolset
from google.adk.tools.openapi_tool.auth.auth_helpers import token_to_scheme_credential
from google.genai import types

### Choose the model

The agent in this tutorial uses Gemini 3.8 Flash. The retry options make the SDK retry a request that fails with a temporary error, such as a 429 or a 500, instead of failing the turn.

In [4]:
model = Gemini(
    model="gemini-3.8-flash",
    retry_options=types.HttpRetryOptions(attempts=4, initial_delay=2.0),
)

## Start the order management API

The mock API is a small FastAPI app in `quickstarts/_services/orders_api`. Start it in the background on port 8010. The API key is `demo-key`; in a real deployment the key comes from Secret Manager.

In [5]:
ORDERS_API_URL = "http://localhost:8010"
ORDERS_API_KEY = "demo-key"

orders_api = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "quickstarts._services.orders_api.app:app", "--port", "8010"],
    cwd=REPO_ROOT,
    env={**os.environ, "ORDERS_API_KEY": ORDERS_API_KEY},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)

Call it directly. Without the key the API refuses the request; with the key it returns the order:

In [6]:
print(httpx.get(f"{ORDERS_API_URL}/orders/BO-000651").status_code)

httpx.get(f"{ORDERS_API_URL}/orders/BO-000651", headers={"X-API-Key": ORDERS_API_KEY}).json()

401


{'order_id': 'BO-000651',
 'store_id': 'S-014',
 'store_name': 'Cymbal Beauty Naperville',
 'status': 'pending',
 'promised_at': '2026-10-03T09:30:00-05:00',
 'ready_at': None,
 'hold_until': None,
 'items': [{'product_id': 'P-0101', 'name': 'Lumière Hydra Cream', 'qty': 1}]}

The request without the key gets `401`. With the key, order BO-000651 at Naperville (S-014) is `pending`, with one Lumière Hydra Cream, promised for 9:30 AM Central time on 3 October 2026. It has no `ready_at` or `hold_until` yet because it is not ready.

## Create the tools

### Tools from the OpenAPI description

`openapi.yaml` in this folder describes the API's two operations, `get_order` and `list_orders`. `token_to_scheme_credential` builds the API-key scheme and credential, and `OpenAPIToolset` turns each operation into a tool that sends the key with every call.

In [7]:
spec = Path("openapi.yaml").read_text()
print(spec[:900])

openapi: 3.0.3
info: {title: Cymbal Beauty Order Management API, version: 1.0.0}
servers: [{url: http://localhost:8010}]
components:
  securitySchemes:
    apiKey: {type: apiKey, in: header, name: X-API-Key}
security: [{apiKey: []}]
paths:
  /orders/{order_id}:
    get:
      operationId: get_order
      summary: Get one buy-online-pick-up-in-store order by id (status, store, items, promised pick-up time, ready time, hold end).
      parameters: [{name: order_id, in: path, required: true, description: "order id like BO-000651", schema: {type: string}}]
      responses: {"200": {description: the order, content: {application/json: {schema: {type: object}}}}}
  /orders:
    get:
      operationId: list_orders
      summary: List a store's buy-online-pick-up-in-store orders, earliest promised pick-up first, optionally filtered by status.
      parameters:
        - {name: store_id, in: query


In [8]:
auth_scheme, auth_credential = token_to_scheme_credential("apikey", "header", "X-API-Key", ORDERS_API_KEY)

orders_toolset = OpenAPIToolset(
    spec_str=spec, spec_str_type="yaml", auth_scheme=auth_scheme, auth_credential=auth_credential
)

for tool in await orders_toolset.get_tools():
    print(f"{tool.name}: {tool.description[:100]}")

get_order: Get one buy-online-pick-up-in-store order by id (status, store, items, promised pick-up time, ready 
list_orders: List a store's buy-online-pick-up-in-store orders, earliest promised pick-up first, optionally filte


### A function tool over the weather API

`get_weather` looks up the city's coordinates, then reads the current temperature, humidity and precipitation. Rain, snow or heat tend to move guests to pick-up orders, so the store team cares about it.

In [9]:
def get_weather(city: str) -> dict:
    """Current temperature (°C), humidity (%) and precipitation (mm) for a city, from the Open-Meteo API."""
    places = httpx.get(
        "https://geocoding-api.open-meteo.com/v1/search", params={"name": city, "count": 1}, timeout=10
    ).json()
    if not places.get("results"):
        return {"status": "ERROR", "error_details": f"no coordinates found for {city!r}"}
    place = places["results"][0]
    current = httpx.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,relative_humidity_2m,precipitation",
        },
        timeout=10,
    ).json()["current"]
    return {
        "status": "SUCCESS",
        "city": place["name"],
        "temperature_c": current["temperature_2m"],
        "humidity_pct": current["relative_humidity_2m"],
        "precipitation_mm": current["precipitation"],
    }


get_weather("Naperville")

{'status': 'SUCCESS',
 'city': 'Naperville',
 'temperature_c': 16.7,
 'humidity_pct': 75,
 'precipitation_mm': 0.0}

The weather comes from the live Open-Meteo API, so your values differ from the ones shown.

## Define the agent

The agent gets both kinds of tool. The `App` adds `ReflectAndRetryToolPlugin`: when a tool call fails, for example with an invalid status value, the plugin gives the error back to the model and lets it correct the call, up to two times.

In [10]:
instruction = """You help Cymbal Beauty store associates with buy-online-pick-up-in-store (BOPIS) orders.
The store is Cymbal Beauty Naperville (S-014) unless the associate names another store id.
- One order: call get_order with its id. Several orders: call list_orders with the store_id and,
  when asked, a status (pending | picked | ready | collected | cancelled).
- Report status and items exactly as returned. Give times in the store's local time with AM/PM and the
  day, for example 9:30 AM Saturday. For a ready order also give hold_until: after that time the order
  is cancelled and its items go back to stock.
- Weather questions: call get_weather with the store's city and say in one sentence what it means for
  pick-up traffic.
- Answer in at most three sentences. Never invent order ids, times, quantities or temperatures."""

agent = LlmAgent(
    name="external_api_agent",
    model=model,
    description="BOPIS order status from the order management API, plus weather for the store's city.",
    instruction=instruction,
    tools=[orders_toolset, get_weather],
)

app = App(name="external_api_agent", root_agent=agent, plugins=[ReflectAndRetryToolPlugin(max_retries=2)])

## Run the agent

Create a runner for the app and a session:

In [11]:
runner = InMemoryRunner(app=app)
session = await runner.session_service.create_session(app_name=app.name, user_id="associate")

Define a helper that sends one message and prints what happened: each tool call with its arguments, and the agent's reply.

In [12]:
async def ask(question: str) -> None:
    """Send one message to the agent and print its tool calls and final answer."""
    message = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=message
    ):
        for call in event.get_function_calls():
            print(f"[{event.author}] calls {call.name}({call.args})")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts if not part.thought)
            if text:
                print(f"\n{text}\n")

Ask about one order. The agent calls the `get_order` tool that `OpenAPIToolset` generated:

In [13]:
await ask("What's the status of pick-up order BO-000651?")

[external_api_agent] calls get_order({'order_id': 'BO-000651'})



Order BO-000651 is currently pending with 1 Lumière Hydra Cream. The promised pick-up time is 9:30 AM Saturday at Cymbal Beauty Naperville.



One `get_order` call. The agent turned `promised_at` (`2026-10-03T09:30:00-05:00`) into 9:30 AM Saturday.

Ask about ready orders and their holds. The agent calls `list_orders` with a status filter:

In [14]:
await ask("Which ready orders at S-014 are close to the end of their hold?")

[external_api_agent] calls list_orders({'store_id': 'S-014', 'status': 'ready'})



Order BO-000664 (1 Glow Body Wash) is closest to the end of its hold, with a hold_until time of 3:00 PM Saturday, after which the order is cancelled and its items go back to stock. Ready order BO-000661 (1 Renew Body Lotion) is held until 2:10 PM Wednesday.



One `list_orders` call with `status: ready`. BO-000664 is held until 3:00 PM on Saturday 3 October and BO-000661 until 2:10 PM on Wednesday 7 October, so BO-000664 comes first. The wording varies from run to run; the orders and times should match.

Ask about the weather:

In [15]:
await ask("What's the weather in Naperville, and does it matter for pick-up today?")

[external_api_agent] calls get_weather({'city': 'Naperville'})



In Naperville, the current temperature is 16.7°C with 75% humidity and 0 mm of precipitation. With mild temperatures and no rain, pick-up traffic should remain steady and unaffected by weather conditions.



The temperature, humidity and precipitation come from the live API and change from run to run, and so does the agent's comment on pick-up traffic.

Try these example phrases:

```
List the pending pick-up orders at S-014
Is order BO-000654 ready?
```

## Run the agent in the ADK developer UI

`agent.py` in this folder defines the same agent and reads `ORDERS_API_KEY` and `ORDERS_API_URL` from the environment. From the repository root, start the mock API, then the developer UI:

```bash
uv run uvicorn quickstarts._services.orders_api.app:app --port 8010 &
echo "ORDERS_API_KEY=demo-key" >> .env
uv run python scripts/quickstart_apps.py 04-external-api-agent
uv run adk web build/quickstart_apps --port 8001
```

On Agent Runtime, deliver `ORDERS_API_KEY` from Secret Manager rather than a plain environment variable.

## Cleaning up

Stop the mock order management API:

In [16]:
orders_api.terminate()

## What's next

- [OpenAPI tools in ADK](https://google.github.io/adk-docs/tools-custom/openapi-tools/)
- [ADK plugins](https://google.github.io/adk-docs/plugins/)
- [Quickstart 05: analyze store data with the BigQuery tools](../05-data-analyst-agent/walkthrough.ipynb)